# Peramalan Harga Sembako & Korelasi dengan Statistik Fintech P2P Lending (OJK)**Tujuan Proyek:**1. Menganalisis tren dan volatilitas harga komoditas pangan pokok nasional (Jan 2024 - Des 2025)2. Mengeksplorasi hubungan antara kondisi pembiayaan fintech P2P lending (OJK) dengan pergerakan harga pangan3. Membangun model peramalan harga sederhana sebagai alat bantu mitigasi risiko kerugian**Sumber Data:**- Harga Pangan: Open Data Badan Pangan Nasional (Rata-rata Harga Pangan Bulanan Tingkat Konsumen Nasional)- Fintech Lending: OJK - Statistik LPBBTI (Layanan Pendanaan Bersama Berbasis Teknologi Informasi)

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport seaborn as snsfrom sklearn.linear_model import LinearRegressionfrom sklearn.metrics import mean_absolute_percentage_error, mean_squared_errorplt.rcParams['figure.figsize'] = (11, 5)plt.rcParams['axes.grid'] = Trueplt.rcParams['grid.alpha'] = 0.3sns.set_palette("Set2")%matplotlib inline

## 1. Load & Persiapan Data

In [ ]:
df = pd.read_csv("../data/master_dataset.csv")bulan_order = ['Januari','Februari','Maret','April','Mei','Juni','Juli',               'Agustus','September','Oktober','November','Desember']df['periode'] = pd.Categorical(df['bulan'], categories=bulan_order, ordered=True)df['bulan_num'] = df['periode'].cat.codes + 1df['tanggal'] = pd.to_datetime(df['tahun'].astype(str) + '-' + df['bulan_num'].astype(str) + '-01')df = df.sort_values('tanggal').reset_index(drop=True)komoditas_cols = ['beras_premium','beras_medium','bawang_merah','bawang_putih',                   'cabai_merah_keriting','cabai_rawit_merah','daging_sapi',                   'daging_ayam','telur_ayam','gula_pasir','minyak_goreng_kemasan']ojk_cols = ['ojk_tkb90','ojk_twp90','ojk_roa','ojk_roe','ojk_bopo','ojk_penyaluran_pertanian_miliar']label_map = {    'beras_premium':'Beras Premium','beras_medium':'Beras Medium','bawang_merah':'Bawang Merah',    'bawang_putih':'Bawang Putih','cabai_merah_keriting':'Cabai Merah Keriting',    'cabai_rawit_merah':'Cabai Rawit Merah','daging_sapi':'Daging Sapi','daging_ayam':'Daging Ayam',    'telur_ayam':'Telur Ayam','gula_pasir':'Gula Pasir','minyak_goreng_kemasan':'Minyak Goreng Kemasan'}print(f"Jumlah baris: {len(df)}")df.head()

## 2. EDA - Tren Harga Seluruh KomoditasMelihat pola pergerakan harga tiap komoditas sepanjang 2 tahun terakhir, termasuk identifikasi pola musiman(misal lonjakan menjelang Ramadan/Lebaran atau akhir tahun).

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(16, 14))axes = axes.flatten()for i, col in enumerate(komoditas_cols):    axes[i].plot(df['tanggal'], df[col], marker='o', markersize=3, linewidth=1.5)    axes[i].set_title(label_map[col], fontsize=11, fontweight='bold')    axes[i].tick_params(axis='x', rotation=45, labelsize=8)    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))axes[-1].axis('off')plt.suptitle('Tren Harga Komoditas Pangan Nasional, Jan 2024 - Des 2025', fontsize=14, fontweight='bold', y=1.01)plt.tight_layout()plt.savefig('../outputs/figures/01_tren_harga_semua_komoditas.png', dpi=140, bbox_inches='tight')plt.show()

## 3. Volatilitas Harga (Coefficient of Variation)Komoditas dengan CV tertinggi adalah yang paling berisiko menyebabkan kerugian akibat fluktuasi harga —prioritas utama untuk dimitigasi dengan model peramalan.

In [ ]:
cv = (df[komoditas_cols].std() / df[komoditas_cols].mean() * 100).sort_values(ascending=False)cv.index = [label_map[c] for c in cv.index]plt.figure(figsize=(10,6))plt.barh(cv.index[::-1], cv.values[::-1], color=sns.color_palette("Set2", len(cv)))plt.xlabel('Coefficient of Variation (%)')plt.title('Volatilitas Harga Komoditas Pangan (Jan 2024 - Des 2025)', fontweight='bold')for i, v in enumerate(cv.values[::-1]):    plt.text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)plt.tight_layout()plt.savefig('../outputs/figures/02_volatilitas_komoditas.png', dpi=140, bbox_inches='tight')plt.show()print("Top 3 komoditas paling volatil:")print(cv.head(3))

## 4. Tren Indikator OJK (Fintech P2P Lending)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)axes[0].plot(df['tanggal'], df['ojk_twp90']*100, marker='o', color='crimson')axes[0].set_title('TWP90 - Tingkat Wanprestasi Fintech P2P Lending (%)', fontweight='bold')axes[0].tick_params(axis='x', rotation=45)axes[1].plot(df['tanggal'], df['ojk_penyaluran_pertanian_miliar'], marker='o', color='seagreen')axes[1].set_title('Penyaluran Pinjaman Fintech ke Sektor Pertanian (Miliar Rp)', fontweight='bold')axes[1].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig('../outputs/figures/03_tren_indikator_ojk.png', dpi=140, bbox_inches='tight')plt.show()

## 5. Analisis Korelasi: Harga Pangan vs Indikator OJK (dengan Lag)Menguji apakah kondisi fintech lending (TWP90, ROA, penyaluran ke sektor pertanian) berkorelasi denganharga pangan pada bulan yang sama (lag=0) atau mendahului 1-2 bulan (lag=1,2) sebagai potential leading indicator.**Catatan penting:** korelasi tidak sama dengan kausalitas, terutama dengan hanya 24 titik data. Temuan dibawah ini bersifat eksploratif, bukan bukti hubungan sebab-akibat yang kuat.

In [ ]:
corr_records = []for col in komoditas_cols:    for ojk_col in ojk_cols:        for lag in [0, 1, 2]:            shifted = df[ojk_col].shift(lag)            valid = shifted.notna()            if valid.sum() > 5:                r = np.corrcoef(df.loc[valid, col], shifted[valid])[0,1]                corr_records.append({'komoditas': label_map[col], 'indikator_ojk': ojk_col, 'lag_bulan': lag, 'korelasi': r})corr_df = pd.DataFrame(corr_records)corr_df.to_csv('../outputs/tabel_korelasi_lengkap.csv', index=False)pivot0 = corr_df[corr_df['lag_bulan']==0].pivot(index='komoditas', columns='indikator_ojk', values='korelasi')plt.figure(figsize=(9,7))sns.heatmap(pivot0, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)plt.title('Korelasi Harga Pangan vs Indikator OJK (tanpa lag)', fontweight='bold')plt.tight_layout()plt.savefig('../outputs/figures/04_heatmap_korelasi.png', dpi=140, bbox_inches='tight')plt.show()

In [ ]:
strongest = corr_df.reindex(corr_df['korelasi'].abs().sort_values(ascending=False).index).head(10)print("Top 10 korelasi terkuat (harga vs OJK, termasuk lag):")strongest

## 6. Peramalan Harga (Baseline Model)Menggunakan **Linear Regression dengan trend + dummy musiman bulanan** sebagai model interpretable,dievaluasi terhadap **naive seasonal baseline**, untuk 2 komoditas paling volatil.> **Catatan keterbatasan:** dengan hanya 24 observasi bulanan, model time-series kompleks (SARIMA/Prophet)> berisiko overfitting. Pendekatan regresi sederhana dipilih agar hasil lebih robust dan mudah diinterpretasi.> Untuk pengembangan lanjutan dengan data historis lebih panjang, SARIMA/SARIMAX (`statsmodels`) atau> Prophet direkomendasikan — lihat bagian "Pengembangan Lanjutan" di README.

In [ ]:
reverse_label = {v:k for k,v in label_map.items()}target_commodities = cv.index[:2].tolist()forecast_results = []fig, axes = plt.subplots(1, 2, figsize=(15,5))for i, target_label in enumerate(target_commodities):    col = reverse_label[target_label]    data = df[['tanggal','bulan_num', col]].copy()    data['t'] = np.arange(len(data))    month_dummies = pd.get_dummies(data['bulan_num'], prefix='m', drop_first=True)    X = pd.concat([data[['t']], month_dummies], axis=1)    y = data[col]    n_test = 6    X_train, X_test = X.iloc[:-n_test], X.iloc[-n_test:]    y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]    model = LinearRegression()    model.fit(X_train, y_train)    y_pred = model.predict(X_test)    naive_pred = y_train.iloc[-n_test:].values if len(y_train) >= n_test else np.repeat(y_train.iloc[-1], n_test)    mape_model = mean_absolute_percentage_error(y_test, y_pred) * 100    rmse_model = np.sqrt(mean_squared_error(y_test, y_pred))    mape_naive = mean_absolute_percentage_error(y_test, naive_pred) * 100    rmse_naive = np.sqrt(mean_squared_error(y_test, naive_pred))    forecast_results.append({        'komoditas': target_label, 'MAPE_model_%': round(mape_model,2), 'RMSE_model': round(rmse_model,1),        'MAPE_naive_%': round(mape_naive,2), 'RMSE_naive': round(rmse_naive,1)    })    axes[i].plot(data['tanggal'], y, label='Aktual', marker='o', markersize=3, color='black')    axes[i].plot(data['tanggal'].iloc[-n_test:], y_pred, label='Prediksi (Model)', marker='s', linestyle='--', color='crimson')    axes[i].axvline(data['tanggal'].iloc[-n_test], color='gray', linestyle=':', alpha=0.7)    axes[i].set_title(f'Forecast: {target_label}', fontweight='bold')    axes[i].legend(fontsize=9)    axes[i].tick_params(axis='x', rotation=45)plt.tight_layout()plt.savefig('../outputs/figures/05_hasil_forecasting.png', dpi=140, bbox_inches='tight')plt.show()forecast_df = pd.DataFrame(forecast_results)forecast_df.to_csv('../outputs/tabel_evaluasi_model.csv', index=False)forecast_df

## 7. Kesimpulan & Temuan Utama1. **Cabai Rawit Merah** dan **Cabai Merah Keriting** adalah komoditas dengan volatilitas harga tertinggi   (CV > 16%) — prioritas utama untuk sistem peringatan dini risiko kerugian.2. Ditemukan korelasi kontemporer yang cukup kuat antara **harga Minyak Goreng Kemasan dengan ROA fintech   lending (r ≈ 0.80)** — namun ini kemungkinan besar korelasi tidak langsung (spurious), keduanya sama-sama   dipengaruhi tren makroekonomi umum, bukan hubungan sebab-akibat langsung.3. Model regresi musiman sederhana **mengungguli baseline naive untuk Cabai Rawit Merah** (MAPE lebih rendah),   namun **kalah dari baseline naive untuk Cabai Merah Keriting** — menunjukkan keterbatasan model dengan   data 24 bulan, dan pentingnya evaluasi jujur per-komoditas, bukan generalisasi berlebihan.4. Hubungan data fintech P2P lending dengan harga pangan lebih bersifat **eksploratif/makro**, karena data   OJK yang tersedia agregat nasional tanpa breakdown sub-sektor pertanian spesifik — ini limitasi data,   bukan kesimpulan bahwa hubungannya tidak ada.## Pengembangan Lanjutan- Perpanjang rentang data historis (>36 bulan) begitu tersedia, untuk model SARIMA/SARIMAX yang lebih robust- Breakdown data fintech per provinsi (Tabel 9 OJK) disandingkan data harga per provinsi untuk analisis regional- Uji Granger causality formal untuk klaim leading indicator yang lebih kuat secara statistik